In [0]:
dbutils.widgets.text("epic_secrets_scope", "", "Epic Secrets Scope")
dbutils.widgets.text("client_id_dbs_key", "", "Epic Client ID DB Secrets Key")
dbutils.widgets.text("token_url", "", "Epic Token URL")
dbutils.widgets.text("algo", "", "Epic Token Enrcyption Algorithm")

In [0]:
widget_values = dbutils.widgets.getAll()
for name, value in widget_values.items():
	print(f"{name} = {value}")

In [0]:
import time
import jwt
import requests

In [0]:
CLIENT_ID = dbutils.secrets.get(scope = widget_values["epic_secrets_scope"], key = widget_values["client_id_dbs_key"])
TOKEN_URL = widget_values["token_url"]
PRIVATE_KEY = dbutils.secrets.get(scope = widget_values["epic_secrets_scope"], key = "private_key")
ALGO = widget_values["algo"]

In [0]:
print(f"""
CLIENT_ID = {CLIENT_ID}
TOKEN_URL = {TOKEN_URL}
PRIVATE_KEY = {PRIVATE_KEY}
ALGO = {ALGO}
""")

In [0]:
def get_access_token():

	now = int(time.time())
	payload = {
		"iss": CLIENT_ID
		, "sub": CLIENT_ID
		, "aud": TOKEN_URL
		, "jti": str(now)
		, "exp": now + 300  # Token valid for 5 minutes
	}

	jwt_token = jwt.encode(payload, PRIVATE_KEY, algorithm=ALGO)

	headers = { "Content-Type": "application/x-www-form-urlencoded" }
	data = {
		"grant_type": "client_credentials"
		, "client_assertion_type": "urn:ietf:params:oauth:client-assertion-type:jwt-bearer"
		, "client_assertion": jwt_token
		, "scope": "system/Patient.read"
	}

	response = requests.post(TOKEN_URL, data=data, headers=headers)
	response.raise_for_status()
	return response.json()["access_token"]

In [0]:
token = get_access_token()
print(token)

In [0]:
token = get_access_token()
headers = {
	"Authorization": f"Bearer {token}"
    ,"Accept": "application/json"
}
url = "https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/Patient/T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"
response = requests.get(url, headers=headers)
response.raise_for_status()
print(response.status_code, response.text)

In [0]:
response.json()